In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MBA-EDA-Cleaning") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setLocalProperty("spark.scheduler.pool", "pool1")
print("SparkSession aktif:", spark.version)

SparkSession aktif: 3.5.0


In [2]:
BRONZE_PATH = "s3a://bigdata/bronze/retail_transactions/retail_transactions.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(BRONZE_PATH)

print(f"Total baris: {df_raw.count()}")
print(f"Kolom: {df_raw.columns}")
df_raw.printSchema()

Total baris: 1000000
Kolom: ['Transaction_ID', 'Date', 'Customer_Name', 'Product', 'Total_Items', 'Total_Cost', 'Payment_Method', 'City', 'Store_Type', 'Discount_Applied', 'Customer_Category', 'Season', 'Promotion']
root
 |-- Transaction_ID: integer (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



In [3]:
print("=== Distribusi Season ===")
df_raw.groupBy("Season").count().orderBy("count", ascending=False).show()

print("=== Distribusi Discount_Applied ===")
df_raw.groupBy("Discount_Applied").count().show()

print("=== Cek Null per Kolom ===")
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
null_counts.show()

print("=== Sample Data ===")
df_raw.select("Transaction_ID", "Product", "Discount_Applied", "Season").show(10)

=== Distribusi Season ===
+------+------+
|Season| count|
+------+------+
|Spring|250368|
|  Fall|250248|
|Winter|249763|
|Summer|249621|
+------+------+

=== Distribusi Discount_Applied ===
+----------------+------+
|Discount_Applied| count|
+----------------+------+
|           false|499896|
|            true|500104|
+----------------+------+

=== Cek Null per Kolom ===
+--------------+----+-------------+-------+-----------+----------+--------------+----+----------+----------------+-----------------+------+---------+
|Transaction_ID|Date|Customer_Name|Product|Total_Items|Total_Cost|Payment_Method|City|Store_Type|Discount_Applied|Customer_Category|Season|Promotion|
+--------------+----+-------------+-------+-----------+----------+--------------+----+----------+----------------+-----------------+------+---------+
|             0|   0|            0|      0|          0|         0|             0|   0|         0|               0|                0|     0|        0|
+--------------+----+----

In [5]:
df_clean = df_raw.select(
    "Transaction_ID",
    "Product",          
    "Discount_Applied",
    "Season"
)

df_clean = df_clean.withColumn(
    "Discount_Applied",
    F.when(F.upper(F.trim(F.col("Discount_Applied"))) == "TRUE", 1)
     .when(F.upper(F.trim(F.col("Discount_Applied"))) == "FALSE", 0)
     .otherwise(None)
     .cast("integer")
)

df_clean = df_clean.dropna(subset=["Transaction_ID", "Product", "Season"])

df_clean = df_clean \
    .withColumn("Season", F.initcap(F.trim(F.col("Season")))) \
    .withColumn("Transaction_ID", F.trim(F.col("Transaction_ID")))

df_clean = df_clean.filter(F.trim(F.col("Product")) != "[]")

print(f"Setelah cleaning: {df_clean.count()} baris")
df_clean.show(5, truncate=False)

Setelah cleaning: 1000000 baris
+--------------+-------------------------------------------------------+----------------+------+
|Transaction_ID|Product                                                |Discount_Applied|Season|
+--------------+-------------------------------------------------------+----------------+------+
|1000000000    |['Ketchup', 'Shaving Cream', 'Light Bulbs']            |1               |Winter|
|1000000001    |['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'Potatoes']|1               |Fall  |
|1000000002    |['Spinach']                                            |1               |Winter|
|1000000003    |['Tissues', 'Mustard']                                 |1               |Spring|
|1000000004    |['Dish Soap']                                          |0               |Winter|
+--------------+-------------------------------------------------------+----------------+------+
only showing top 5 rows



In [6]:
df_dedup = df_clean.dropDuplicates(["Transaction_ID", "Product"])
removed = df_clean.count() - df_dedup.count()
print(f"Duplikat dihapus: {removed} baris")
df_clean = df_dedup

Duplikat dihapus: 0 baris


In [7]:
SILVER_PATH = "s3a://bigdata/silver/retail_cleaned/"

df_clean.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH)

print(f"Silver layer tersimpan di: {SILVER_PATH}")
df_clean.printSchema()

Silver layer tersimpan di: s3a://bigdata/silver/retail_cleaned/
root
 |-- Transaction_ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Discount_Applied: integer (nullable = true)
 |-- Season: string (nullable = true)

